# Partitioning uncertainty by year instead of GWL

In [1]:
import xarray as xr
import xagg as xa
import numpy as np
import pandas as pd
import geopandas as gpd
import os
import re
import glob
import copy
import zarr
import warnings
from tqdm.notebook import tqdm
from dask.diagnostics import ProgressBar
from matplotlib import colors as mcolors

import distributed

from funcs_support import get_filepaths,get_params,utility_save
from funcs_readregs import coeffs_mortality, coeffs_gdppc
from funcs_processing import calc_uncerts
from funcs_plot import plot_map, vals_to_color_triangle

dir_list = get_params()

In [2]:
from distributed import Client

# Start dask client 
client = Client()
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/8787/status,Workers: 6
Total threads: 18,Total memory: 72.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:41051,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:33333,Total threads: 3
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/39129/status,Memory: 12.00 GiB
Nanny: tcp://127.0.0.1:38669,


2026-04-13 18:42:12,238 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle c74b91696109eaec2481bd3b41872e9d initialized by task ('rechunk-merge-rechunk-transfer-8c82ed3802f983923bbc5f3fb35218a1', 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 9) executed on worker tcp://127.0.0.1:45253
2026-04-13 18:42:18,231 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle c74b91696109eaec2481bd3b41872e9d deactivated due to stimulus 'task-finished-1776127338.2306788'


In [3]:
params_proc = {'mortality':{'varname':'tassumpoly','geo_typ':'impact-regions','popwght':False},
               'gdppc':{'varname':'tassumpoly','geo_typ':'adm1','popwght':True},
              'payroll':{'varname':'tasmaxbinF','geo_typ':'adm2-us','popwght':True}}

#params_proc = {'payroll':{'varname':'tasmaxbinF','geo_typ':'adm2-us','popwght':True}}


In [4]:
overwrite = True
ssp = 'ssp245'
for proc in ['mortality']:#params_proc:
    output_fn = f'{dir_list['proc']}uncert_parts/uncert_parts_{proc}_byyr.zarr'
    if overwrite or (not os.path.exists(output_fn)):
        #---------------------------
        # Setup
        #---------------------------
        # Load impacts projections
        fns = np.sort(glob.glob(f'{dir_list['proc']}impact_projs/{proc}/d{proc}*_agg-1deg_*ALLTIMEFRAMES*.zarr'))
        def load_impact(fn):
            ds_tmp = xr.open_zarr(fn,consolidated=False)
            ds_tmp = ds_tmp.set_coords('locid') # Should be coordinate, instead of concatenated variable
            ds_tmp = ds_tmp.assign_coords({'model':(('idv'),[re.split(r'\_',re.split(r'\/',fn)[-1])[2]]*ds_tmp.sizes['idv'])})
        
            ds_tmp['exprun'] = ds_tmp['experiment']+ds_tmp['run']
            ds_tmp = ds_tmp.reset_index('idv')
            ds_tmp = ds_tmp.set_index(idv = ['model','exprun'])
            return ds_tmp
        
        with warnings.catch_warnings():
            # Silence zarr warnings related to the /.done flags used
            warnings.filterwarnings('ignore')
            dss = xr.concat([load_impact(fn) for fn in fns],
                        dim='idv',join='outer')

        # SUBSET TO DESIRED SSP 
        dss = dss.isel(idv = (dss.experiment.load() == ssp))
        
        dss = dss.chunk({'poly_idx':50,'idv':-1,'proj_base':-1,'gwl':-1})

        # Save temporary rechunked file to make dask graphing more stable
        tmp_fn = dir_list['proc']+'tmp/uncert_rechunk_cache_tmp.zarr'
        dss = dss.reset_index('idv')
        for v in dss.coords:
            dss[v] = dss[v].load()
        utility_save(dss.drop_encoding(),tmp_fn,zarr_mode = 'w')
        dss = xr.open_zarr(tmp_fn)
        for v in ['model','exprun']:
            # Explicitly cast these dims as objects because of a numpy bug
            dss[v] = dss[v].astype(object)
        dss = dss.set_index(idv = ['model','exprun'])
        
        # Figure out how many BCD-ME members reach a given GWL
        # (copy.deepcopy() necessary as a workaround around a numpy bug
        # affecting the unstack a few lines down that's triggered 
        # indirectly by the reset_index here)
        gwl_counter = copy.deepcopy(dss.maxgwl).load().reset_index('idv').groupby(['model','maxgwl']).count()
        gwl_counter = gwl_counter.sel(maxgwl = gwl_counter.maxgwl.values[::-1]).cumsum('maxgwl')
        #print(gwl_counter)
        
        uncerts_all = []
        #---------------------------
        # Uncertainty partitioning over all runs
        #---------------------------
        dss_tmp = dss.unstack('idv')['d'+proc]
        dss_tmp = dss_tmp.chunk({'model':-1,'exprun':-1}).unify_chunks()
        
        # Calculate uncertainty partitioning
        with warnings.catch_warnings():
            # Ignoring chunk increase warnings and divide by 0s in uncertainty 
            # partitioning (happens over ocean, e.g.)
            warnings.filterwarnings('ignore')
            uncerts = xr.map_blocks(calc_uncerts,dss_tmp).compute()

        uncerts['numruns'] = (~np.isnan(dss_tmp.mean('poly_idx'))).sum('exprun').load()
        uncerts = uncerts.expand_dims({'sample':['allruns']})
        
        uncerts_all.append(uncerts)


        #---------------------------
        # Uncertainty partitioning over only models with > 1 run
        #---------------------------

        nruns = 3
        gwlmods = gwl_counter.sel(maxgwl=2070)
        proc_mods = gwlmods.isel(model = gwlmods>=nruns).model.values
        
        dss_tmp = dss.unstack('idv')['d'+proc]
        dss_tmp = dss_tmp.sel(model=proc_mods)
        dss_tmp = dss_tmp.chunk({'model':-1,'exprun':-1}).unify_chunks()
        
        # Calculate uncertainty partitioning
        with warnings.catch_warnings():
            # Ignoring chunk increase warnings and divide by 0s in uncertainty 
            # partitioning (happens over ocean, e.g.)
            warnings.filterwarnings('ignore')
            uncerts = xr.map_blocks(calc_uncerts,dss_tmp).compute()

        uncerts['numruns'] = (~np.isnan(dss_tmp.mean('poly_idx'))).sum('exprun').load()
        uncerts = uncerts.expand_dims({'sample':[f'{str(nruns)}runs{str(dss_tmp.sizes['model'])}mods']})
        
        uncerts_all.append(uncerts)

        #---------------------------
        # Uncertainty partitioning over only models with > 1 run,
        # but allowing different models over different timeperiods
        # if they fit the run limit
        #---------------------------
        nruns = 3
        dss_tmp = dss.unstack('idv')['d'+proc]
        dss_tmp = dss_tmp.chunk({'model':-1,'exprun':-1}).unify_chunks()
        
        dss_tmp = dss_tmp.where((~np.isnan(dss_tmp)).sum('exprun')>=nruns)
        
        # Calculate uncertainty partitioning
        with warnings.catch_warnings():
            # Ignoring chunk increase warnings and divide by 0s in uncertainty 
            # partitioning (happens over ocean, e.g.)
            warnings.filterwarnings('ignore')
            uncerts = xr.map_blocks(calc_uncerts,dss_tmp).compute()

        uncerts['numruns'] = (~np.isnan(dss_tmp.mean('poly_idx'))).sum('exprun').load()
        uncerts = uncerts.expand_dims({'sample':[f'atleast{str(nruns)}runs']})
        
        uncerts_all.append(uncerts)
        
        #---------------------------
        # Export
        #---------------------------
        # Concatenate into single ds
        uncerts_all = xr.concat(uncerts_all,dim='sample')
        uncerts_all = uncerts_all.set_coords('numruns')
        uncerts_all = uncerts_all.where(uncerts_all.sum('typ')>0)
        
        # Export
        uncerts_all.attrs = {'SOURCE':'4d_partition_uncertainty_byyr.ipynb',
                             'DESCRIPTION':f'Uncertainty partitioning of projections of {proc} between model, internal, and reanalysis sources'}
        
        utility_save(uncerts_all,output_fn,zarr_mode = 'w')

        os.system('rm -rf '+tmp_fn)
        print('Temporary file '+tmp_fn+' removed.')
    else:
        print(output_fn+' exists!')
    


/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=6, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=13, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning i

/glade/derecho/scratch/schwarzwald/impact_uncertainty/tmp/uncert_rechunk_cache_tmp.zarr saved!


/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of fre

/glade/derecho/scratch/schwarzwald/impact_uncertainty/uncert_parts/uncert_parts_mortality_byyr.zarr removed to allow overwrite!


/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=6, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/impact_uncertainty/uncert_parts/uncert_parts_mortality_byyr.zarr saved!
Temporary file /glade/derecho/scratch/schwarzwald/impact_uncertainty/tmp/uncert_rechunk_cache_tmp.zarr removed.


In [ ]:
## Note: TaiESM1 is the odd one out on payroll that's causing model unc to blow up with the all runs 

In [70]:
df = get_filepaths(source_dir = 'proc',dir_list = {'proc':'/glade/derecho/scratch/schwarzwald/bcd_me/'})

In [77]:
xr.open_zarr('/glade/campaign/cgd/cas/schwarzwald/bcd_me/bcd_me_qdm-byyr_CanESM5.zarr').has_data.sel(gwl=2040).load()

<xarray.DataArray 'has_data' (variable: 2, idv: 14)> Size: 28B
array([[ True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True, False,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True, False]])
Coordinates:
  * variable    (variable) object 16B 'tasmax' 'tas'
    experiment  (idv) StringDType() 224B 'ssp245' 'ssp245' ... 'ssp245' 'ssp245'
    has_data    (variable, idv) bool 28B True True True True ... True True False
    model       (idv) <U7 392B 'CanESM5' 'CanESM5' ... 'CanESM5' 'CanESM5'
    run         (idv) StringDType() 224B 'r10i1p1f1' 'r10i1p2f1' ... 'r1i1p1f1'
    gwl         float64 8B 2.04e+03
Dimensions without coordinates: idv

In [83]:
xr.open_zarr('/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tasmaxbinF_agg-1deg_CanESM5_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr').has_data.sel(gwl=2).load()

<xarray.DataArray 'has_data' (variable: 2, idv: 87)> Size: 174B
array([[ True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True, False,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True],
       [False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False,  True,  True, False, False, False,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True, False,  True, False, False,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True, False,  True, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False]])
Coordinates:
  * variable    (variable) object 16B 'tas' 'tasmax'
  * idv         (idv) int64 696B 0 1 2 3 4 5 6 7 8 ... 79 80 81 82 83 84 85 86
    experiment  (idv) <U6 2kB 'ssp245' 'ssp245' 'ssp245' ... 'ssp585' 'ssp585'
    has_data    (variable, idv) bool 174B True True True ... False False False
    model       (idv) <U7 2kB 'CanESM5' 'CanESM5' ... 'CanESM5' 'CanESM5'
    run         (idv) <U9 3kB 'r10i1p1f1' 'r10i1p2f1' ... 'r9i1p1f1' 'r9i1p2f1'
    gwl         float64 8B 2.0

In [74]:
df.query('model == "CanESM5" and exp == "ssp245" and (varname == "tas" or varname == "tasmax") and gwl == "ALLTIMEPERIODS"').groupby(['varname','proj_base']).count()

freq  model  exp  run  gwl  proj_method  dwnscl_method  \
varname proj_base                                                           
tas     ERA5-025     13     13   13   13   13           13              0   
        GMFD         13     13   13   13   13           13              0   
        JRA-3Q       13     13   13   13   13           13              0   
        MERRA2       13     13   13   13   13           13              0   
tasmax  ERA5-025     13     13   13   13   13           13              0   
        GMFD         13     13   13   13   13           13              0   
        JRA-3Q       13     13   13   13   13           13              0   
        MERRA2       13     13   13   13   13           13              0   

                   dwnscl_target  filetype  path  suffix  time  unlabeled  \
varname proj_base                                                           
tas     ERA5-025               0        13    13       0     0          0   
        GMFD                   0        13    13       0     0          0   
        JRA-3Q                 0        13    13       0     0          0   
        MERRA2                 0        13    13       0     0          0   
tasmax  ERA5-025               0        13    13       0     0          0   
        GMFD                   0        13    13       0     0          0   
        JRA-3Q                 0        13    13       0     0          0   
        MERRA2                 0        13    13       0     0          0   

                   mtime  
varname proj_base         
tas     ERA5-025      13  
        GMFD          13  
        JRA-3Q        13  
        MERRA2        13  
tasmax  ERA5-025      13  
        GMFD          13  
        JRA-3Q        13  
        MERRA2        13

In [60]:
dss.isel(resolution=1,timeframe=0,obstreat=0,weighting=0).sel(model='CanESM5').mean('poly_idx').dpayroll.load()

<xarray.DataArray 'dpayroll' (proj_base: 4, gwl: 4, exprun: 13)> Size: 2kB
array([[[ 0.        ,  0.        ,  0.        ,         nan,
                 nan,         nan,         nan,         nan,
                 nan,         nan,         nan,         nan,
                 nan],
        [-0.00714487, -0.00789808, -0.00538173,         nan,
                 nan,         nan,         nan,         nan,
                 nan,         nan,         nan,         nan,
                 nan],
        [-0.01819681, -0.01374254, -0.01391854,         nan,
                 nan,         nan,         nan,         nan,
                 nan,         nan,         nan,         nan,
                 nan],
        [-0.0250433 , -0.02350797, -0.02254265,         nan,
                 nan,         nan,         nan,         nan,
                 nan,         nan,         nan,         nan,
                 nan]],

       [[ 0.        ,  0.        ,         nan,         nan,
                 nan,         nan,         nan,         nan,
                 nan,         nan,         nan,         nan,
...
                 nan,         nan,         nan,         nan,
                 nan,         nan,         nan,         nan,
                 nan]],

       [[ 0.        ,  0.        ,  0.        ,         nan,
                 nan,         nan,         nan,         nan,
                 nan,         nan,         nan,         nan,
                 nan],
        [-0.00448643, -0.00491763, -0.00336243,         nan,
                 nan,         nan,         nan,         nan,
                 nan,         nan,         nan,         nan,
                 nan],
        [-0.01158269, -0.00866743, -0.00896422,         nan,
                 nan,         nan,         nan,         nan,
                 nan,         nan,         nan,         nan,
                 nan],
        [-0.01578006, -0.0150267 , -0.01445349,         nan,
                 nan,         nan,         nan,         nan,
                 nan,         nan,         nan,         nan,
                 nan]]])
Coordinates:
  * proj_base   (proj_base) <U6 96B 'ERA5' 'MERRA2' 'JRA-3Q' 'GMFD'
  * gwl         (gwl) float64 32B 0.61 2.01e+03 2.04e+03 2.07e+03
  * exprun      (exprun) object 104B 'ssp245r10i1p1f1' ... 'ssp245r16i1p1f1'
    experiment  (exprun) StringDType() 208B 'ssp245' 'ssp245' ... 'ssp245'
    maxgwl      (exprun) float64 104B 2.07e+03 2.07e+03 2.07e+03 ... nan nan nan
    run         (exprun) StringDType() 208B 'r10i1p1f1' ... 'r16i1p1f1'
    obstreat    <U3 12B 'sep'
    resolution  <U4 16B '1deg'
    timeframe   <U9 36B '1986-2008'
    weighting   <U3 12B 'pop'
    model       <U7 28B 'CanESM5'

In [55]:
dss_tmp.sel(model='CanESM5').mean('poly_idx').isel(resolution=1,timeframe=0,obstreat=0,weighting=0,gwl=0).load()

<xarray.DataArray 'dpayroll' (proj_base: 4, exprun: 38)> Size: 1kB
array([[ 0., nan,  0.,  0., nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan],
       [ 0., nan,  0., nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan],
       [ 0., nan,  0., nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan],
       [ 0., nan,  0.,  0., nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]])
Coordinates:
  * proj_base   (proj_base) <U6 96B 'ERA5' 'MERRA2' 'JRA-3Q' 'GMFD'
  * exprun      (exprun) object 304B 'ssp245r10i1p1f1' ... 'ssp245r9i1p1f2'
    experiment  (exprun) object 304B 'ssp245' nan 'ssp245' ... nan nan nan
    maxgwl      (exprun) float64 304B 2.07e+03 nan 2.07e+03 ... nan nan nan
    run         (exprun) object 304B 'r10i1p1f1' nan 'r10i1p2f1' ... nan nan nan
    model       <U7 28B 'CanESM5'
    gwl         float64 8B 0.61
    obstreat    <U3 12B 'sep'
    resolution  <U4 16B '1deg'
    timeframe   <U9 36B '1986-2008'
    weighting   <U3 12B 'pop'

In [53]:
# Load impacts projections# Load impacts projections
fns = np.sort(glob.glob(f'{dir_list['proc']}impact_projs/{proc}/d{proc}*_agg-1deg_*ALLTIMEFRAMES*.zarr'))
def load_impact(fn):
    ds_tmp = xr.open_zarr(fn,consolidated=False)
    ds_tmp = ds_tmp.set_coords('locid') # Should be coordinate, instead of concatenated variable
    ds_tmp = ds_tmp.assign_coords({'model':(('idv'),[re.split(r'\_',re.split(r'\/',fn)[-1])[2]]*ds_tmp.sizes['idv'])})

    ds_tmp['exprun'] = ds_tmp['experiment']+ds_tmp['run']
    ds_tmp = ds_tmp.reset_index('idv')
    ds_tmp = ds_tmp.set_index(idv = ['model','exprun'])
    return ds_tmp

with warnings.catch_warnings():
    # Silence zarr warnings related to the /.done flags used
    warnings.filterwarnings('ignore')
    dss = xr.concat([load_impact(fn) for fn in fns],
                dim='idv',join='outer')

# SUBSET TO DESIRED SSP 
dss = dss.isel(idv = (dss.experiment.load() == ssp))

dss = dss.chunk({'poly_idx':50,'idv':-1,'proj_base':-1,'gwl':-1})

# Save temporary rechunked file to make dask graphing more stable
tmp_fn = dir_list['proc']+'tmp/uncert_rechunk_cache_tmp.zarr'
dss = dss.reset_index('idv')
for v in dss.coords:
    dss[v] = dss[v].load()
utility_save(dss.drop_encoding(),tmp_fn,zarr_mode = 'w')
dss = xr.open_zarr(tmp_fn)
for v in ['model','exprun']:
    # Explicitly cast these dims as objects because of a numpy bug
    dss[v] = dss[v].astype(object)
dss = dss.set_index(idv = ['model','exprun'])

# Figure out how many BCD-ME members reach a given GWL
# (copy.deepcopy() necessary as a workaround around a numpy bug
# affecting the unstack a few lines down that's triggered 
# indirectly by the reset_index here)
gwl_counter = copy.deepcopy(dss.maxgwl).load().reset_index('idv').groupby(['model','maxgwl']).count()
gwl_counter = gwl_counter.sel(maxgwl = gwl_counter.maxgwl.values[::-1]).cumsum('maxgwl')
#print(gwl_counter)

uncerts_all = []
#---------------------------
# Uncertainty partitioning over all runs
#---------------------------
dss_tmp = dss.unstack('idv')['d'+proc]
dss_tmp = dss_tmp.chunk({'model':-1,'exprun':-1}).unify_chunks()


/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=13, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=6, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning i

/glade/derecho/scratch/schwarzwald/impact_uncertainty/tmp/uncert_rechunk_cache_tmp.zarr saved!


In [26]:
(~np.isnan(dss_tmp.mean('poly_idx'))).sum('exprun').load()

<xarray.DataArray 'dpayroll' (proj_base: 4, resolution: 2, timeframe: 2,
                              obstreat: 1, weighting: 2, gwl: 4, model: 10)> Size: 10kB
array([[[[[[[ 1,  9, 13, ..., 19, 10,  5],
            [ 1,  9, 13, ..., 19, 10,  5],
            [ 1,  9, 13, ..., 19, 10,  5],
            [ 1,  9, 13, ..., 19, 10,  5]],

           [[ 1,  9, 13, ..., 19, 10,  5],
            [ 1,  9, 13, ..., 19, 10,  5],
            [ 1,  9, 13, ..., 19, 10,  5],
            [ 1,  9, 13, ..., 19, 10,  5]]]],



         [[[[ 1,  9, 13, ..., 19, 10,  5],
            [ 1,  9, 13, ..., 19, 10,  5],
            [ 1,  9, 13, ..., 19, 10,  5],
            [ 1,  9, 13, ..., 19, 10,  5]],

           [[ 1,  9, 13, ..., 19, 10,  5],
            [ 1,  9, 13, ..., 19, 10,  5],
            [ 1,  9, 13, ..., 19, 10,  5],
...
            [ 1,  9, 13, ..., 19, 10,  5],
            [ 1,  9, 13, ..., 19, 10,  5]],

           [[ 1,  9, 13, ..., 19, 10,  5],
            [ 1,  9, 13, ..., 19, 10,  5],
            [ 1,  9, 13, ..., 19, 10,  5],
            [ 1,  9, 13, ..., 19, 10,  5]]]],



         [[[[ 1,  9, 13, ..., 19, 10,  5],
            [ 1,  9, 13, ..., 19, 10,  5],
            [ 1,  9, 13, ..., 19, 10,  5],
            [ 1,  9, 13, ..., 19, 10,  5]],

           [[ 1,  9, 13, ..., 19, 10,  5],
            [ 1,  9, 13, ..., 19, 10,  5],
            [ 1,  9, 13, ..., 19, 10,  5],
            [ 1,  9, 13, ..., 19, 10,  5]]]]]]],
      shape=(4, 2, 2, 1, 2, 4, 10))
Coordinates:
  * proj_base   (proj_base) <U6 96B 'ERA5' 'MERRA2' 'JRA-3Q' 'GMFD'
  * resolution  (resolution) object 16B '025deg' '1deg'
  * timeframe   (timeframe) object 16B '1986-2008' 'ALLYEARS'
  * obstreat    (obstreat) object 8B 'sep'
  * weighting   (weighting) object 16B 'pop' 'raw'
  * gwl         (gwl) float64 32B 0.61 2.01e+03 2.04e+03 2.07e+03
  * model       (model) object 80B 'CNRM-CM6-1' 'CNRM-ESM2-1' ... 'MRI-ESM2-0'